<p style='
  color: #3b4045; 
  text-align: center;
  font-weight: bold;
  font-family: -apple-system,BlinkMacSystemFont, "Segoe UI Adjusted","Segoe UI","Liberation Sans",sans-serif;     font-size: 2.07692308rem; '> 
    Vascular Graph Extraction
</p>


In [1]:
%load_ext autoreload
%autoreload 2

from jppype import Mosaic, imshow, vscode_theme

from fundus_toolkits import FundusData
from fundus_vessels_toolkit.models import segment_vessels
from fundus_vessels_toolkit.pipelines import FundusVesselSegToGraph

vscode_theme()

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [ ]:
# Path to the raw fundus image
RAW_PATH = "/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/Fundus-AV/1-images/046_N.png"  # "PATH/TO/FUNDUS/IMAGE"

# Path to a vessel ground truth (for comparison purposes)
VESSELS_PATH = (
    "/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/Fundus-AV/2-av-pred_FVT/046_N.png"  # "PATH/TO/VESSEL/GROUND/TRUTH"
)


### Load the raw Image and the Vascular Segmentation


Load image and segmentation mask


In [12]:
fundus = FundusData(image=RAW_PATH, av=VESSELS_PATH)
raw = fundus.image
true_vessels = fundus.vessels

ValueError: Could not load image from /run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/FundusAV/1-images/046_N.png

Perform segmentation using a pretrained model


In [4]:
vessels = segment_vessels(fundus.image)
fundus_pred = fundus.update(vessels=vessels)

---


### Skeletonize and Extract Graph


All the skeletonization and graph extraction pipeline is accessible through `RetinalVesselSeg2Graph` objects.

This object tunes all the parameters of the pipeline according to a single parameter: the maximum vessel diameter.
Apriori, a good approximation is a 80th of the image width (13px for a 1024x1024 px image).


In [5]:
max_vessel_diameter = fundus.shape[1] // 80
seg2graph = FundusVesselSegToGraph(max_vessel_diameter)
seg2graphFVT = FundusVesselSegToGraph(max_vessel_diameter)
seg2graphFVT.skeletonize_method = "fvt"

To compute the graph extraction you simply need to call the object with a segmentation map:


In [6]:
av = fundus.av
graph_fvt = seg2graphFVT(av)
graph = seg2graph(vessels)

%timeit seg2graphFVT.skeletonize(av)
%timeit seg2graph.skeletonize(vessels)

19.3 ms ± 622 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
41.1 ms ± 181 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


The returned object is a `VascularGraph` which expose all the properties of the extracted graph.

- The list of branches : `vgrapp.branch_list`
- The y and x position of the nodes: `vgraph.node_coord()`

This object is also usefull to cast the graph to other representation. For example:

- Node Adjacency Matrix: `vgraph.node_adjacency_matrix()`
- Branch Adjacency Matrix: `vgraph.branch_adjacency_matrix()`


#### Display


Lets compare the graph extracted on the predicted segmentation and on the ground truth segmentation side by side!

_(The prediction is on the left, the ground truth on the right.)_


In [8]:
from fundus_vessels_toolkit.vascular_data_objects.vgraph import VGraph


def setup_view(view, vessels_map, seg2graph):
    view.add_label(vessels_map, "vessel", opacity=0.3)
    view.add_label(seg2graph.skeletonize(vessels_map), "skel", "grey", opacity=0.5)
    view["vessel graph"] = seg2graph(vessels_map).jppype_layer(edge_map=True, edge_labels=False, interpolate=False)


mosaic = Mosaic(2, background=raw, cell_height=650, sync=True, cols_titles=["IA Segmentation", "Ground Truth"])
setup_view(mosaic[0], true_vessels, seg2graph)
setup_view(mosaic[1], fundus.av, seg2graphFVT)

mosaic.show()

GridBox(children=(HTML(value='<h3 style="text-align: center;">IA Segmentation</h3>'), HTML(value='<h3 style="t…

---


### Performing only the skeletonization


The method `seg2graph()` actually perform the graph extraction in three steps:

- Skeletonization (morphological reduction of the segmentation to the vessels centerline): `seg2graph.skeletonize(vessel_segmentation_map)`
- Graph extraction (identification of branches and nodes in the vascular skeleton and connection parsing): `seg2graph.skel_to_vgraph(vessel_skeleton_map, vessel_segmentation_map)`.
- Graph simplification (removal of small branches, merging close nodes...): `seg2graph.simplify_vgraph(vgraph)`

Those method can be called individually, for example to get the result of the skeletonization step only.


In [8]:
skeleton = seg2graph.skeletonize(fundus.artery_map)

You may notice some difference between the skeleton obtained through `seg2graph.skeletonize(vessels)` and by direct call of `seg2graph(vessels)`, because several topological simplification is performed on the latter (merging close nodes, removing small spurs or cycles...).


In [9]:
v = imshow(raw)
v.add_label(vessels, "vessel", "white", options={"opacity": 0.2})
v.add_label(skeleton, "skeleton", {1: "white", 2: "red", 3: "green"})

v.show()

View2D()